# 03 — Conditionals (Questions → Decisions)
### Python Basics · Part 3 of 5

The gate at Iron Gym has about one second to decide: **let in**, **turn away**, or **let in with a warning**. Conditionals are how code makes that decision — and the interesting part isn't the `if` keyword. It's **which question gets asked first**.

## The core idea: first match wins

Python reads an `if / elif / else` chain top to bottom and takes **the first branch whose condition is truthy**. Everything below it is skipped, even if it would also have been true. So in a conditional chain, **order is logic**.

| Tool | Use it when | Watch out for |
|---|---|---|
| `if / elif / else` | choosing **one** outcome among several | the order of the branches |
| separate `if`s | several **independent** checks that can all fire | they don't exclude each other |
| ternary `a if cond else b` | choosing one **value** | nesting them |
| `match / case` | branching on the **shape** of data (commands, patterns) | Python 3.10+ only |

Because we want to run the *same* decision on many inputs, this notebook wraps decisions in tiny **functions** (`def ...`). A function is just a named block of code that takes inputs and `return`s a result — the full story is Day 10.

## Setup

In [1]:
import sys
assert sys.version_info >= (3, 10), "match-case needs Python 3.10+"
print("ready - Python", sys.version.split()[0])

ready - Python 3.12.3


---
## 1. Order is logic — the bug that never crashes

Iron Gym plans: **Basic ≥ ₹500**, **Pro ≥ ₹1000**, **Elite ≥ ₹2000**. Here's the version most beginners write first:

In [2]:
def plan_for_budget_buggy(budget):
    if budget >= 500:
        return "Basic"
    elif budget >= 1000:          # can NEVER run for 1000+ : the first branch already caught it
        return "Pro"
    elif budget >= 2000:
        return "Elite"
    return "No plan"

def plan_for_budget(budget):
    if budget >= 2000:            # most demanding condition FIRST
        return "Elite"
    elif budget >= 1000:
        return "Pro"
    elif budget >= 500:
        return "Basic"
    return "No plan"

for b in (300, 500, 1000, 2500):
    print(f"₹{b:<5} buggy: {plan_for_budget_buggy(b):<8} fixed: {plan_for_budget(b)}")

₹300   buggy: No plan  fixed: No plan
₹500   buggy: Basic    fixed: Basic
₹1000  buggy: Basic    fixed: Pro
₹2500  buggy: Basic    fixed: Elite


No error, no warning — the buggy version just sells every rich member the cheapest plan. **Rule:** when conditions overlap, put the *most specific / most demanding* one first.

---
## 2. Flat beats nested — guard clauses

Deep nesting makes you hold five conditions in your head at once. The fix: check the **bad cases first and leave early**, so the happy path reads straight down.

In [3]:
def gate_nested(is_member, days_left, checked_in):
    if is_member:
        if days_left >= 0:
            if not checked_in:
                return "ALLOW"
            else:
                return "DENY: already inside"
        else:
            return "DENY: membership expired"
    else:
        return "DENY: not a member"

def gate_flat(is_member, days_left, checked_in):
    if not is_member:
        return "DENY: not a member"
    if days_left < 0:
        return "DENY: membership expired"
    if checked_in:
        return "DENY: already inside"
    return "ALLOW"

scenarios = [(False, 10, False), (True, -2, False), (True, 10, True), (True, 10, False)]
for s in scenarios:                       # *s unpacks the tuple into the three arguments
    assert gate_nested(*s) == gate_flat(*s)
    print(f"{str(s):<22} -> {gate_flat(*s)}")

(False, 10, False)     -> DENY: not a member
(True, -2, False)      -> DENY: membership expired
(True, 10, True)       -> DENY: already inside
(True, 10, False)      -> ALLOW


Same behaviour, half the indentation. Each `if` in `gate_flat` removes one failure case, and the last line is the success path.

---
## 3. Truthiness in conditions — and the `0` trap

`if x:` works on *anything* (non-empty → true). It's idiomatic for "is this list empty?" — and **wrong** when `0` or `""` is a legitimate value.

In [4]:
members = []
if not members:                         # idiomatic emptiness check (better than len(members) == 0)
    print("nobody checked in yet")

def visits_message(visits):
    if not visits:                      # BUG: 0 visits is real data, not "missing"
        return "No data"
    return f"{visits} visits"

def visits_message_fixed(visits):
    if visits is None:                  # only None means "missing"
        return "No data"
    return f"{visits} visits"

print(visits_message(0), "|", visits_message_fixed(0), "|", visits_message_fixed(None))

nobody checked in yet
No data | 0 visits | No data


**Rule:** use `if x:` to ask *"is there anything here?"*; use `if x is None:` to ask *"was a value ever given?"*. A member with 0 visits this month is information — your gym owner wants to know about them.

---
## 4. The ternary — choose a *value*, not a path

When all you need is one value or the other, a conditional expression keeps it on one line:

In [5]:
days_left = 12
status = "Active" if days_left >= 0 else "Expired"
print(status)

# Nesting works but gets unreadable fast - past one level, use a regular if/elif:
age = 17
ticket = "free" if age < 5 else "child" if age < 18 else "adult"
print(ticket)

Active
child


---
## 5. `match` / `case` — branching on the *shape* of data (Python 3.10+)

An `if`/`elif` chain asks yes/no questions. `match` asks **"what does this look like?"** — and can pull pieces out as it goes. It shines for commands, tokens and structured input. The front desk types short commands:

In [6]:
def desk_command(text):
    match text.lower().split():
        case ["checkin", member_id]:                       # exactly 2 words, first is 'checkin'
            return f"Checking in {member_id.upper()}"
        case ["renew", member_id, months] if months.isdigit():   # a guard: extra condition
            return f"Renewing {member_id.upper()} for {months} months"
        case ["renew", member_id, _]:                      # _ = "anything"
            return "Months must be a number"
        case ["help"] | ["?"]:                             # | = either pattern
            return "Commands: checkin <id>, renew <id> <months>"
        case _:                                            # default branch
            return "Unknown command"

for cmd in ["checkin IG-42", "renew IG-42 6", "renew IG-42 six", "help", "?", "dance"]:
    print(f"{cmd:<18} -> {desk_command(cmd)}")

checkin IG-42      -> Checking in IG-42
renew IG-42 6      -> Renewing IG-42 for 6 months
renew IG-42 six    -> Months must be a number
help               -> Commands: checkin <id>, renew <id> <months>
?                  -> Commands: checkin <id>, renew <id> <months>
dance              -> Unknown command


In [7]:
def day_type(day):
    match day.lower():
        case "sat" | "sun":
            return "weekend"
        case "mon" | "tue" | "wed" | "thu" | "fri":
            return "weekday"
        case _:
            return "invalid"

for d in ("Sat", "WED", "xyz"):
    print(d, "->", day_type(d))

Sat -> weekend
WED -> weekday
xyz -> invalid


> For plain value comparisons `match` is just a tidier `if/elif`. Reach for it when the **structure** matters (lists of a certain length, a leading keyword, nested data).

---
## 6. Combining conditions & independent `if`s

Use `and`, `or`, `not`, `in` to build one precise question. And remember: a chain of **separate** `if`s is not the same as `elif` — every `if` that's true will fire.

In [8]:
plan, days_left, hour, frozen = "pro", 200, 6, False

if plan in ("pro", "elite") and days_left > 0 and not frozen:
    print("full access")

perks = []                                  # independent checks: several can be true at once
if plan in ("pro", "elite"):
    perks.append("steam room")
if days_left > 180:
    perks.append("free towel")
if hour < 7:
    perks.append("early-bird locker")
print(perks)

full access
['steam room', 'free towel', 'early-bird locker']


---
## Real-world capstone: the Iron Gym gate

One function, six rules, **order = priority**. A rule earlier in the chain overrides everything after it — so the owner's priorities live in the order of the `if`s.

1. no plan → deny   2. expired → deny   3. already in today → deny
4. closed (outside 05:00–23:00) → deny   5. **basic** plan during peak (17:00–20:59) → deny
6. otherwise allow — with a renewal warning if ≤ 7 days remain

In [9]:
def gate_decision(plan, days_left, checked_in_today, hour):
    """Return (decision, reason). The ORDER of the checks is the priority."""
    if plan is None:
        return "DENY", "not a member"
    if days_left < 0:
        return "DENY", f"expired {-days_left} day(s) ago"
    if checked_in_today:
        return "DENY", "already checked in today"
    if not (5 <= hour < 23):
        return "DENY", "gym is closed (05:00-23:00)"
    if plan == "basic" and 17 <= hour < 21:
        return "DENY", "basic plan: no peak-hour entry"
    if days_left <= 7:
        return "ALLOW", f"renew soon - {days_left} day(s) left"
    return "ALLOW", "welcome"

tests = [
    ("walk-in, no plan",        (None,    0,   False, 10), "DENY"),
    ("expired last week",       ("pro",  -7,   False, 10), "DENY"),
    ("second entry today",      ("pro",  30,   True,  10), "DENY"),
    ("2 a.m.",                  ("elite", 90,  False, 2),  "DENY"),
    ("basic at 6 p.m.",         ("basic", 90,  False, 18), "DENY"),
    ("pro at 6 p.m.",           ("pro",   90,  False, 18), "ALLOW"),
    ("renewal due in 3 days",   ("basic", 3,   False, 9),  "ALLOW"),
    ("regular morning",         ("pro",   120, False, 7),  "ALLOW"),
]
print(f"{'scenario':<24}{'decision':<8} reason")
print("-" * 62)
for label, args, expected in tests:
    decision, reason = gate_decision(*args)
    assert decision == expected, f"{label}: expected {expected}, got {decision}"
    print(f"{label:<24}{decision:<8} {reason}")

scenario                decision reason
--------------------------------------------------------------
walk-in, no plan        DENY     not a member
expired last week       DENY     expired 7 day(s) ago
second entry today      DENY     already checked in today
2 a.m.                  DENY     gym is closed (05:00-23:00)
basic at 6 p.m.         DENY     basic plan: no peak-hour entry
pro at 6 p.m.           ALLOW    welcome
renewal due in 3 days   ALLOW    renew soon - 3 day(s) left
regular morning         ALLOW    welcome


### Try it yourself
1. Add a rule: a **frozen** membership (a new boolean argument) is denied with the reason `"membership frozen"`. Where in the chain must it go so an expired-*and*-frozen member gets the more useful message?
2. **Swap** rules 3 and 4 (already-in vs closed) and re-run the table. Which scenarios change — and does any assertion fail? That is exactly what "order is logic" means.
3. Rewrite the final `if days_left <= 7` using a **ternary** that returns the tuple directly.
4. Replace `plan == "basic"` with `plan not in ("pro", "elite")`. What new plan name would now be blocked at peak time by accident? (Think about a `"student"` plan.)
5. Rebuild `desk_command` as an `if/elif` chain using `startswith` and `len(words)`. Which version is easier to extend with a new `"freeze <id> <days>"` command?

---
## Quick reference

| I want to… | Use |
|---|---|
| choose one outcome from several | `if / elif / else` — put the most specific condition first |
| flatten nested ifs | guard clauses: check failures first, `return` early |
| ask "is it empty?" | `if not items:` |
| ask "was a value given?" | `if x is None:` (not `if not x:`) |
| pick one of two values | `a if cond else b` |
| branch on the shape of text/data | `match … case …` |
| check "one of these" | `x in (a, b, c)` |

**Rule of thumb:** *write the checks in the order you'd want the denial message to appear — the first matching rule wins.*

---
## 🎯 Interview corner

**1. `elif` vs several `if`s?** `elif` is mutually exclusive (first match wins); separate `if`s are independent and can all run.
**2. What counts as falsy?** `None`, `False`, zero of any numeric type, and empty containers/strings. Everything else is truthy.
**3. `if x:` vs `if x is None:`?** The first also rejects `0` and `""`; the second rejects only `None`.
**4. What's a guard clause?** An early `return` (or `continue`) for an invalid case so the main logic stays un-nested.
**5. When would you use `match` over `if/elif`?** When branching on structure — sequence length, a leading keyword, nested data — not just on equality.
**6. Ternary syntax?** `value_if_true if condition else value_if_false`.

**Next:** `04_loops/` — doing the same thing for *every* member without copy-pasting.